# Train from your manually sorted folders — no labeling interface

On your computer create **BabyTraining/back**, **BabyTraining/side**, **BabyTraining/prone**, and optional **BabyTraining/unknown**. Copy about 100 varied images into each posture folder to start, keep filenames/image contents, then compress the parent folder into **BabyTraining.zip**. Do not create train/test folders.

Your folder choices are the labels. This notebook accepts that ZIP, extracts MediaPipe's original **12 metrics**, trains **ONE posture classifier**, reports measured results, and can export the existing live box/risk/alarm handoff. It does not use MobileNet or notebook review widgets.

**A small pilot is not proof of 95%.** Posture risk is not an airway/SIDS probability. [Full manual-folder instructions](https://github.com/kg905/SeniorDesignProject/blob/main/docs/MANUAL_FOLDERS.md). Choose a CPU runtime and run cells in order.


## 1. Get the code and original grouping reference
A sparse checkout avoids downloading all 5,814 source images again. Existing checkouts are fast-forwarded without deleting local data.

In [ ]:
import os, subprocess, sys
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
GIT_ENV = dict(os.environ, GIT_LFS_SKIP_SMUDGE='1')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/kg905/SeniorDesignProject.git', str(ROOT)], env=GIT_ENV, check=True)
    subprocess.run(['git', 'sparse-checkout', 'set', 'src', 'training', 'deploy/single-model',
                    'docs', 'models', 'artifacts/dataset_audit_v2'], cwd=ROOT, env=GIT_ENV, check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=ROOT, env=GIT_ENV, check=True)
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
print('Source ready:', ROOT)


## 2. Install dependencies

This affects this temporary Colab runtime, not your Pi. Keep only one pip cv2 distribution. If Colab requests a restart, restart after installation, rerun section 1, then continue to section 3. Do not skip an unsuccessful install.


In [ ]:
%pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless
%pip install -r /content/SeniorDesignProject/training/requirements.txt
%pip install --no-deps -e /content/SeniorDesignProject
!apt-get update -qq
!apt-get install -y -qq git-lfs libgles2 libegl1 libgl1 libportaudio2


## 3. Upload your sorted ZIP to LOCAL Colab storage

Upload BabyTraining.zip, not the original unsorted Roboflow ZIP. Upload, extraction and training stay under **/content/cribwatch-runs**, not the Google Drive mount. This prevents a disconnected Drive from interrupting active file writes. **Local files disappear on a runtime reset: download your model/results in section 8 before closing/resetting Colab.**

If the ZIP is already in Drive, set ZIP_IN_DRIVE below; only that option mounts Drive and copies the source ZIP locally. Otherwise leave it empty and select the ZIP from your computer. Existing Drive reviews/previous runs are not overwritten.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
ROOT = Path('/content/SeniorDesignProject')
os.chdir(ROOT)
import importlib
sys.path.insert(0, str(ROOT / "src"))
importlib.invalidate_caches()
from google.colab import files
from cribwatch.colab_io import local_zip_run
ZIP_IN_DRIVE = ''  # optional: '/content/drive/MyDrive/BabyTraining.zip'
if ZIP_IN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
RUN, ZIP_PATH = local_zip_run(source_zip=ZIP_IN_DRIVE or None)
print('Verified local ZIP:', ZIP_PATH)
print('LOCAL run:', RUN)
print('Download section 8 outputs before resetting/disconnecting this runtime.')


## 4. Import folders as labels and check counts

The importer accepts one enclosing BabyTraining folder (or direct class folders). It verifies images, preserves your labels, keeps unknowns out of posture training, and quarantines conflicting copies. Hash matches to the original audit preserve known duplicate/source groups; new images get conservative duplicate grouping. None of this invents baby IDs.


In [ ]:
from cribwatch.assets import safe_unzip
from cribwatch.folder_data import import_folders
from cribwatch.data import read_csv
EXTRACTED = RUN / 'sorted-images'
safe_unzip(ZIP_PATH, EXTRACTED)
AUDIT = RUN / 'folder-audit'
summary = import_folders(EXTRACTED, AUDIT, reference_index=ROOT / 'artifacts/dataset_audit_v2/images.csv')
print('Your folder counts:', summary['folder_counts'])
print('Eligible exact-unique posture images:', summary['eligible_unique_counts'])
print('Matched to original audit:', summary['matched_original_images'])
print('Conservative source groups:', summary['source_groups'])
print('Conflicting images excluded:', summary['conflicting_images'])
print('Ignored files:', summary['ignored_files'][:10])
if summary['conflicting_images']:
    print('Inspect', AUDIT / 'conflicts.json', 'and correct the computer folders for a future run.')
if any(summary['eligible_unique_counts'].get(label, 0) == 0 for label in ['back','side','prone']):
    raise ValueError('Need non-conflicting images in ALL THREE posture folders before training.')
MANIFEST = AUDIT / 'images.csv'
print('Labels saved:', MANIFEST)


## 5. Verify MediaPipe assets and extract all 12 metrics
Only the two official pose/face assets are needed. They are fetched through Git LFS and checked against pinned SHA256 hashes. No labeling widgets are involved.

In [ ]:
import hashlib
from cribwatch.assets import ASSETS
subprocess.run(['git', 'lfs', 'install', '--local'], cwd=ROOT, check=True)
subprocess.run(['git', 'lfs', 'pull', '--include=models/*.task', '--exclude='], cwd=ROOT, check=True)
for name in ['pose_landmarker_lite.task','face_landmarker.task']:
    assert hashlib.sha256((ROOT / 'models' / name).read_bytes()).hexdigest() == ASSETS[name][1], name
from cribwatch.data import extract_manifest
FEATURES = RUN / 'features.csv'
extract_manifest(MANIFEST, FEATURES, ROOT / 'models', workers=1)
rows = read_csv(FEATURES)
eligible = [r for r in rows if r['training_eligible'] == '1']
coverage = sum(r['pose_ok'] == '1' for r in eligible) / len(eligible)
print(f'Usable pose coverage on eligible images: {coverage:.1%}')
for label in ['back','side','prone']:
    selected = [r for r in eligible if r['posture'] == label]
    usable = sum(r['pose_ok'] == '1' for r in selected)
    groups = len({r['source_group'] for r in selected})
    print(label, ':', len(selected), 'images,', usable, 'usable poses,', groups, 'source groups')
if coverage < .95:
    print('Pose coverage is below 95%. Better labels alone cannot yield 95% all-image accuracy under this gate.')


## 6. Train ONE classifier and read the actual results

Default `source_groups` is an exploratory pilot. Code makes about 60/20/20 train/validation/test splits, keeping known related images together. It needs enough groups/classes; many augmented copies of one image do not supply that diversity. A split error means add varied sources, not bypass the guard.

Only use `baby_ids` with actual identities and recording metadata in optional metadata.csv; never invent them. A final claim requires a fresh independently reviewed held-out cohort. Unknown predictions count as errors. Do not repeatedly tune against a final test score.


In [ ]:
from cribwatch.learning import train
EVALUATION_MODE = 'source_groups'
if EVALUATION_MODE not in ['source_groups', 'baby_ids']:
    raise ValueError('Use source_groups or baby_ids')
MODEL = RUN / 'model'
report = train(FEATURES, MODEL, exploratory=(EVALUATION_MODE == 'source_groups'))
print('Overall accuracy (unknown = error):', f"{report['accuracy_all_images_unknown_counts_as_error']:.1%}")
print('Prediction coverage:', f"{report['coverage']:.1%}")
for label, values in report['per_class'].items():
    print(label, 'recall:', f"{values['recall']:.1%}")
print('95% group-bootstrap interval:', report['accuracy_95ci_group_cluster_bootstrap'])
print('Report:', MODEL / 'test.json')
if not report['research_95pct_point_target_met']:
    print('95% TARGET NOT ESTABLISHED. This is a research pilot, not a validated infant-safety result.')


## 7. See example scans with a pose box and risk level

These are examples after evaluation, not a new accuracy test. Body position maps to lower/moderate/high posture risk; unknown stays unknown. The visible-landmark box is not a separate baby/airway detector. Confidence is not a medical-risk percentage.


In [ ]:
examples = [r['image_path'] for r in read_csv(MODEL / 'test_predictions.csv')][:6]
ANNOTATED = RUN / 'annotated-scans'
subprocess.run([sys.executable, '-m', 'cribwatch.showcase', 'image', '--model', str(MODEL / 'posture.joblib'),
                '--models', str(ROOT / 'models'), '--showcase', '--output', str(ANNOTATED), *examples], check=True)
from IPython.display import display
from PIL import Image
for image in sorted(ANNOTATED.glob('*.jpg')):
    display(Image.open(image))


## 8. DOWNLOAD the trained/evaluated Pi package before ending the runtime

This packages one classifier, MediaPipe assets, report, scan/receiver scripts and instructions. Local results do not survive a runtime reset. Download the ZIP and keep it on your computer. Optional backup to Drive runs only if BACKUP_TO_DRIVE is enabled below, after local processing is complete; a Drive backup failure does not remove your local outputs.

Physical Pi camera/GPIO/Wi-Fi and continuous-event tests remain necessary. No 95% or V10 artifact is fabricated.


In [ ]:
ZIP_OUT = RUN / 'SIDS_Pi5_Deployment.zip'
subprocess.run([sys.executable, str(ROOT / 'deploy/single-model/build_handoff.py'),
                '--model', str(MODEL), '--output', str(ZIP_OUT)], check=True)
files.download(str(ZIP_OUT))
print('Local download source:', ZIP_OUT)
print('Keep the downloaded ZIP on your computer; /content storage is temporary.')
BACKUP_TO_DRIVE = False
if BACKUP_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive', force_remount=True)
    backup = Path('/content/drive/MyDrive/CribWatchManual') / RUN.name
    backup.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_OUT, backup / ZIP_OUT.name)
    print('Copied to Drive:', backup / ZIP_OUT.name)


## Adding more images later

Add photos to the same class folders on your computer, create a ZIP of the entire updated BabyTraining folder, and start a new run. The new run imports your current folder choices and leaves old local runs intact while the runtime exists; download results before resetting; adding files does not automatically retrain a deployed model. The original review-widget CSV is not needed for this workflow.

After installation on the Pi desktop, the package supports:

```bash
python monitor_pi5.py --model models --models models --picamera --display --showcase --alarm-risk moderate
```

Adopt the bundled TCP 5000 Pi 4 receiver and shared-key contract before adding `--alarm-host <Pi4-IP>`. Verify actual sound, risk timing, latching/mute/acknowledgement and camera/network faults. The attached PDF's V10 notebook/receiver source were not supplied; compatibility with unseen code is not claimed. Live screenshots demonstrate operation; only independent evaluation can support a 95% posture claim. No SIDS-prevention claim is made.
